# Linear Regression in R (Assignment 3, Part A)

This notebook walks through a simple linear regression analysis step by step.

**What we will do:**
1. Read `regression_data.csv`
2. Make a scatter plot
3. Fit a linear regression model
4. Overlay the regression line on the plot
5. Calculate and print diagnostics (slope, intercept, correlation, MSE, R-squared)
6. Save the final plot

> **Tip:** The companion script `r_lin_reg.r` does the same work from the command line:
> `Rscript r_lin_reg.r regression_data.csv YearsExperience Salary`

In [ ]:
# Load ggplot2 for nicer plots
library(ggplot2)

# Column names and data file (change these to analyze different columns)
FILENAME <- "regression_data.csv"
X_COLUMN <- "YearsExperience"
Y_COLUMN <- "Salary"

## Step 1: Read the CSV file

A CSV is a simple spreadsheet saved as text. `read.csv()` loads it into a data frame.

In [ ]:
data <- read.csv(FILENAME)
head(data)

## Step 2: Scatter plot

Each dot is one row in the dataset. This helps us see whether a straight-line trend exists.

In [ ]:
plot(data[[X_COLUMN]], data[[Y_COLUMN]],
     col = "red", pch = 19,
     main = paste(Y_COLUMN, "vs", X_COLUMN),
     xlab = X_COLUMN,
     ylab = Y_COLUMN)

## Step 3: Fit a linear regression model

The `lm()` function finds the best straight line:

$$y = \text{slope} \times x + \text{intercept}$$

In [ ]:
formula <- as.formula(paste(Y_COLUMN, "~", X_COLUMN))
model <- lm(formula, data = data)

slope <- coef(model)[2]
intercept <- coef(model)[1]
predictions <- predict(model, newdata = data)

cat("Slope:    ", round(slope, 4), "\n", sep = "")
cat("Intercept:", round(intercept, 4), "\n", sep = "")

## Step 4: Overlay the regression line

The blue line shows the model's predicted values for each x value.

In [ ]:
ggplot(data, aes(x = .data[[X_COLUMN]], y = .data[[Y_COLUMN]])) +
  geom_point(color = "red") +
  geom_line(aes(y = predictions), color = "blue", linewidth = 1) +
  labs(
    title = paste("Linear Regression:", Y_COLUMN, "vs", X_COLUMN),
    x = X_COLUMN,
    y = Y_COLUMN
  ) +
  theme_minimal()

## Step 5: Model diagnostics

- **Correlation (r):** strength and direction of the linear relationship
- **MSE:** average squared prediction error (lower is better)
- **R-squared:** proportion of variance explained by the model (closer to 1 is better)

In [ ]:
correlation <- cor(data[[X_COLUMN]], data[[Y_COLUMN]])
mse <- mean((data[[Y_COLUMN]] - predictions)^2)
r_squared <- summary(model)$r.squared

cat("Correlation:", round(correlation, 4), "\n")
cat("MSE:        ", round(mse, 2), "\n", sep = "")
cat("R-squared:  ", round(r_squared, 4), "\n", sep = "")

## Step 6: Final plot with statistics and saved output

We add the equation and diagnostics to the plot, then save it as a PNG file.

In [ ]:
stats_label <- paste0(
  "y = ", round(slope, 2), "x + ", round(intercept, 2),
  "\nr = ", round(correlation, 4),
  "\nMSE = ", round(mse, 2),
  "\nR² = ", round(r_squared, 4)
)

final_plot <- ggplot(data, aes(x = .data[[X_COLUMN]], y = .data[[Y_COLUMN]])) +
  geom_point(color = "red", alpha = 0.85) +
  geom_line(aes(y = predictions), color = "blue", linewidth = 1) +
  annotate(
    "text",
    x = min(data[[X_COLUMN]]),
    y = max(data[[Y_COLUMN]]) - 0.15 * (max(data[[Y_COLUMN]]) - min(data[[Y_COLUMN]])),
    label = stats_label,
    hjust = 0,
    size = 3.5
  ) +
  labs(
    title = paste("Linear Regression:", Y_COLUMN, "vs", X_COLUMN),
    x = X_COLUMN,
    y = Y_COLUMN
  ) +
  theme_minimal()

print(final_plot)

output_file <- "linear_regression_r_output.png"
ggsave(output_file, plot = final_plot, width = 8, height = 6, dpi = 150)
cat("Plot saved to", output_file, "\n")

## Interpretation (example for this dataset)

On `regression_data.csv`, salary tends to increase with years of experience.
The slope tells us the expected salary increase per additional year, and the intercept
is the predicted salary when experience is zero. A strong positive correlation and
a reasonably high R-squared suggest the straight-line model captures much of the pattern.